# Paso 2 -- Secuencia de experimentos: forma de la recompensa

Antes de gastar el unico entrenamiento largo del ciclo (`01_enfoque_y_entrenamiento_final.ipynb`),
se explora METODICAMENTE la forma de la recompensa sobre `escalon_1` (franja
manana, 2 barcos) -- entrenamientos CORTOS (`timesteps_exploracion` del
config, ~7 min cada uno) para poder iterar rapido. La infraestructura de
entrenamiento (`VecNormalize`, entropia, frecuencia de actualizacion --
`agente.hiperparametros`) ya esta validada de una ronda de diagnostico
anterior y NO se vuelve a tocar aca -- lo unico que cambia entre
experimentos es la recompensa.

**La secuencia, y la pregunta que responde cada paso:**
- **A** -- solo penalizacion de tiempo, CON techo por persona (el valor de
  produccion). ¿Que tan bien sirve el agente con la recompensa mas simple
  posible?
- **B** -- lo mismo, pero SIN techo por persona. Hipotesis: mejor que A,
  porque sin techo el agente siente mas la urgencia de atender primero a
  quien lleva mas tiempo esperando -- con techo, alguien que espera 35 min y
  alguien que espera 50 min penalizan exactamente igual (los dos llegaron al
  maximo), asi que la recompensa ya no distingue entre ellos; sin techo, el
  de 50 min sigue pesando mas.
- **C1/C2** -- sobre el ganador de A/B, se agrega una penalizacion de
  movimiento PEQUENA (0.003 y 0.001). Analisis de escala antes de elegir el
  numero: la penalizacion de una persona va de 0 a 1 (el techo, si esta
  activo) segun (sobrante/18)^2 -- alguien con apenas ~13 min de espera ya
  penaliza (1/18)^2 ≈ 0.003, asi que una penalizacion de movimiento del
  mismo orden (0.003) o mas chica (0.001) es necesaria para que mover un
  barco no pese, en la recompensa, tanto como dejar a alguien esperando --
  si no, el agente podria preferir quedarse quieto para ahorrarse el costo
  de moverse, aunque eso signifique dejar gente sin atender. El objetivo
  real de este paso: entregar a la gente en menos de 12 minutos (la
  tolerancia, ver `simulador/README.md`) usando los menos movimientos
  posibles -- ni tan poca penalizacion que el movimiento sea gratis, ni
  tanta que domine sobre la incomodidad.

Cada experimento se evalua contra la politica base sobre las mismas 5
semillas de evaluacion de siempre (`agente.evaluacion.semillas`, fuera del
rango de entrenamiento).

In [3]:
%load_ext autoreload
%autoreload 2

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

BASE_DIR = Path.cwd().parent
SIMULADOR_DIR = (BASE_DIR / "../simulador").resolve()
POLITICA_BASE_DIR = (BASE_DIR / "../politica_base").resolve()
BB_DIR = (BASE_DIR / "../bergen-boats").resolve()
DEMAND_DIR = (BASE_DIR / "../demand").resolve()
sys.path.insert(0, str(DEMAND_DIR / "src"))
sys.path.insert(0, str(SIMULADOR_DIR / "src"))
sys.path.insert(0, str(POLITICA_BASE_DIR / "src"))
sys.path.insert(0, str(BASE_DIR / "src"))

import masas as demand_masas
import llegadas as demand_llegadas
from entrenamiento import EntornoDemandaAleatoria
from politica_base import asignar_flota
import metricas as met

from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

cfg_sim = yaml.safe_load(open(SIMULADOR_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_bb = yaml.safe_load(open(BB_DIR / "config" / "instance.yaml", encoding="utf-8"))
cfg_demand = demand_masas.cargar_config(DEMAND_DIR / "config" / "instance.yaml")

resumen_masas = pd.read_csv(DEMAND_DIR / "output" / "masas_por_nodo.csv", index_col="id")
intensidad_od = pd.read_csv(DEMAND_DIR / "output" / "matriz_intensidad_od.csv")
poblacion_total_zonas = resumen_masas["poblacion_total"].sum()
conexiones_fuertes = cfg_bb["garantia"]["conexiones_fuertes"]
nodos = [n["id"] for n in cfg_bb["nodos_demanda"]]
matriz_tiempos = pd.read_csv(BB_DIR / "02_ruteo_navegable" / "output" / "matriz_tiempos_min.csv", index_col=0)

cfg_agente = cfg_sim["agente"]
cfg_hiper = cfg_agente["hiperparametros"]
cfg_exp = cfg_agente["experimentos"]
cfg_escalon = cfg_sim["escalones"][cfg_exp["escalon_base"]]
hora_ini_min, hora_fin_min = cfg_escalon["horas"][0] * 60, cfg_escalon["horas"][1] * 60
semillas_eval = cfg_agente["evaluacion"]["semillas"]

OUT_DIR = BASE_DIR / "output" / "experimentos"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Instancia: {cfg_exp['escalon_base']} ({cfg_escalon['num_barcos']} barcos, {cfg_escalon['horas']}h)")
print(f"timesteps_exploracion: {cfg_exp['timesteps_exploracion']}")
print(f"Hiperparametros PPO (compartidos, no varian entre experimentos): {cfg_hiper}")
print(f"Semillas de evaluacion: {semillas_eval}")


def construir_entorno(cfg_recompensa, semilla_entrenamiento):
    return EntornoDemandaAleatoria(
        generar_llegadas_dia_fn=demand_llegadas.generar_llegadas_dia,
        cfg_demand=cfg_demand, intensidad_od=intensidad_od, conexiones_fuertes=conexiones_fuertes,
        poblacion_total_zonas=poblacion_total_zonas, horas=cfg_escalon["horas"],
        porcentaje_poblacion_dia=cfg_escalon["porcentaje_poblacion_dia"],
        semilla_entrenamiento=semilla_entrenamiento,
        matriz_tiempos=matriz_tiempos, nodos=nodos, num_barcos=cfg_escalon["num_barcos"],
        capacidad_barco=cfg_bb["flota"]["capacidad_pasajeros"], nodo_inicial=cfg_bb["flota"]["nodo_inicial"],
        paso_tiempo_min=cfg_sim["paso_tiempo_min"], hora_inicio_min=hora_ini_min, hora_fin_min=hora_fin_min,
        cfg_recompensa=cfg_recompensa, unidad_demanda=cfg_sim["unidad_demanda"],
    )


Instancia: escalon_1 (2 barcos, [6, 9]h)
timesteps_exploracion: 30000
Hiperparametros PPO (compartidos, no varian entre experimentos): {'usar_vecnormalize': True, 'ent_coef': 0.01, 'learning_rate': 0.0003, 'n_steps': 512}
Semillas de evaluacion: [1001, 1002, 1003, 1004, 1005]


## Política base sobre las 5 semillas de evaluación (referencia fija para todos los experimentos)

In [4]:
def correr_base(cfg_recompensa, semilla):
    env = construir_entorno(cfg_recompensa, semilla_entrenamiento=None)
    obs, info = env.reset(seed=semilla)
    reward_total = 0.0
    while True:
        estado = info["_estado_obj"]
        libres = [b for b in estado.barcos if b.libre]
        decisiones = asignar_flota(libres, estado, matriz_tiempos, env.capacidad_barco, cfg_recompensa)
        accion = np.array([
            env.codificar_accion_barco(decisiones[b.id]) if b.libre else 0
            for b in estado.barcos
        ])
        obs, r, terminated, truncated, info = env.step(accion)
        reward_total += r
        if truncated or terminated:
            break
    return env, reward_total


# La base no depende de los overrides de recompensa que se prueban por experimento --
# usa siempre la formula de PRODUCCION (recompensa: del config, sin overrides), la
# misma que reportan los escalones 1-3. Se corre UNA vez y se reusa como referencia
# fija para comparar cada experimento.
envs_base_ref = [correr_base(cfg_sim["recompensa"], s) for s in semillas_eval]
corrida_base_ref = met.combinar_corridas([e for e, r in envs_base_ref])
reporte_base_ref = met.reporte_completo(corrida_base_ref)
reward_base_ref = sum(r for e, r in envs_base_ref)

print(f"Politica base (5 semillas): {reporte_base_ref['globales']['pct_atendidas']:.1f}% atendidas, "
      f"reward total (formula de produccion) = {reward_base_ref:.1f}")


Politica base (5 semillas): 88.0% atendidas, reward total (formula de produccion) = -5259.0


## Función reusable: entrenar + evaluar un experimento

Entrena PPO con `timesteps_exploracion` sobre `cfg_recompensa`, evalúa el
agente resultante sobre las 5 semillas (usando la MISMA `cfg_recompensa`
del experimento, para que el reward del agente sea comparable contra sí
mismo entre pasos), y junta las métricas completas
(`metricas.combinar_corridas` + `reporte_completo`).

In [5]:
def entrenar_evaluar_experimento(nombre, cfg_recompensa):
    t0 = time.time()
    venv = DummyVecEnv([lambda: Monitor(
        construir_entorno(cfg_recompensa, semilla_entrenamiento=cfg_exp["semilla_entrenamiento"]),
        filename=str(OUT_DIR / f"monitor_{nombre}"),
    )])
    venv = VecNormalize(venv, norm_obs=True, norm_reward=True, gamma=cfg_agente["gamma"])
    model = PPO(
        "MlpPolicy", venv, gamma=cfg_agente["gamma"],
        ent_coef=cfg_hiper.get("ent_coef", 0.0), learning_rate=cfg_hiper.get("learning_rate", 3e-4),
        n_steps=cfg_hiper.get("n_steps", 2048), seed=cfg_exp["semilla_entrenamiento"], verbose=0,
    )
    model.learn(total_timesteps=cfg_exp["timesteps_exploracion"])
    dt = time.time() - t0

    venv_stats = venv
    venv_stats.training = False
    venv_stats.norm_reward = False

    def correr_agente(semilla):
        env = construir_entorno(cfg_recompensa, semilla_entrenamiento=None)
        obs, info = env.reset(seed=semilla)
        reward_total = 0.0
        while True:
            obs_norm = venv_stats.normalize_obs(obs)
            accion, _ = model.predict(obs_norm, deterministic=True)
            obs, r, terminated, truncated, info = env.step(accion)
            reward_total += r
            if truncated or terminated:
                break
        return env, reward_total

    envs_agente = [correr_agente(s) for s in semillas_eval]
    corrida_agente = met.combinar_corridas([e for e, r in envs_agente])
    reporte_agente = met.reporte_completo(corrida_agente)
    reward_agente = sum(r for e, r in envs_agente)

    resultado = {
        "experimento": nombre,
        "cfg_recompensa": cfg_recompensa,
        "pct_atendidas": reporte_agente["globales"]["pct_atendidas"],
        "espera_media_min": reporte_agente["globales"]["espera_media_min"],
        "sistema_medio_min": reporte_agente["globales"]["sistema_medio_min"],
        "sistema_maximo_min": reporte_agente["globales"]["sistema_maximo_min"],
        "movimientos_totales": int(reporte_agente["por_barco"]["movimientos"].sum()),
        "ocupacion_media": float(reporte_agente["por_barco"]["ocupacion_media"].mean()),
        "reward_total": reward_agente,
        "tiempo_entrenamiento_s": dt,
        "conservacion_ok": reporte_agente["conservacion"]["cuadra"],
    }
    print(f"[{nombre}] {dt:.0f}s -- {resultado['pct_atendidas']:.1f}% atendidas "
          f"(base: {reporte_base_ref['globales']['pct_atendidas']:.1f}%), "
          f"espera media {resultado['espera_media_min']:.1f} min, "
          f"{resultado['movimientos_totales']} movimientos, reward={resultado['reward_total']:.1f}")
    return resultado, model, reporte_agente


resultados = []


## Experimento A -- solo penalización de tiempo, con techo por persona

In [6]:
cfg_A = dict(cfg_sim["recompensa"])
cfg_A.update(cfg_exp["A"]["recompensa_overrides"])
print("Config A:", cfg_exp["A"]["descripcion"])
print("  ", cfg_exp["A"]["recompensa_overrides"])
res_A, model_A, rep_A = entrenar_evaluar_experimento("A", cfg_A)
resultados.append(res_A)


Config A: Solo penalizacion de tiempo, CON techo por persona (el default de produccion)
   {'tolerancia_incomodidad_min': 0, 'sobrante_normalizador_min': 1, 'peso_movimiento': 0.0, 'premio_por_persona_entregada': 0.0, 'penalizacion_maxima_persona': 1000000.0}
[A] 149s -- 60.5% atendidas (base: 88.0%), espera media 22.2 min, 144 movimientos, reward=-40550002.6


## Experimento B -- solo penalización de tiempo, SIN techo por persona

In [7]:
cfg_B = dict(cfg_sim["recompensa"])
cfg_B.update(cfg_exp["B"]["recompensa_overrides"])
print("Config B:", cfg_exp["B"]["descripcion"])
print("  ", cfg_exp["B"]["recompensa_overrides"])
res_B, model_B, rep_B = entrenar_evaluar_experimento("B", cfg_B)
resultados.append(res_B)


Config B: Solo penalizacion de tiempo, SIN techo por persona -- hipotesis: mejor que A, porque sin techo el agente siente mas la urgencia de atender primero a quien lleva mas tiempo esperando (35 min y 50 min ya no penalizan igual)
   {'tolerancia_incomodidad_min': 12, 'sobrante_normalizador_min': 1, 'peso_movimiento': 0.0, 'premio_por_persona_entregada': 0.0, 'penalizacion_maxima_persona': 1000000.0}
[B] 149s -- 49.2% atendidas (base: 88.0%), espera media 32.6 min, 108 movimientos, reward=-36433170.6


## A vs. B -- ¿se confirma la hipótesis?

La hipótesis era que B (sin techo) sirve mejor que A (con techo), porque
sin techo el agente distingue entre alguien muy atrasado y alguien
levemente atrasado en vez de tratarlos igual una vez ambos llegan al
máximo.

In [8]:
comparacion_AB = pd.DataFrame([res_A, res_B])[[
    "experimento", "pct_atendidas", "espera_media_min", "sistema_medio_min",
    "sistema_maximo_min", "movimientos_totales", "reward_total",
]]
display(comparacion_AB.round(2))

ganador_nombre = "B" if res_B["pct_atendidas"] >= res_A["pct_atendidas"] else "A"
cfg_ganador_AB = cfg_B if ganador_nombre == "B" else cfg_A
print(f"\nGanador de A/B: {ganador_nombre} "
      f"({'confirma' if ganador_nombre == 'B' else 'CONTRADICE'} la hipotesis) "
      f"-- {resultados[-1 if ganador_nombre=='B' else -2]['pct_atendidas']:.1f}% atendidas")


,experimento,pct_atendidas,espera_media_min,sistema_medio_min,sistema_maximo_min,movimientos_totales,reward_total
0,A,60.47,22.25,30.80,96.62,144,-40550002.58
1,B,49.18,32.63,46.18,111.13,108,-36433170.64



Ganador de A/B: A (CONTRADICE la hipotesis) -- 60.5% atendidas


## Experimento C1/C2 -- sobre el ganador de A/B, penalización de movimiento pequeña

In [9]:
cfg_C1 = dict(cfg_ganador_AB)
cfg_C1.update(cfg_exp["C1"]["recompensa_overrides"])
print("Config C1:", cfg_exp["C1"]["descripcion"])
print("  peso_movimiento:", cfg_C1["peso_movimiento"], "| techo:", cfg_C1["penalizacion_maxima_persona"])
res_C1, model_C1, rep_C1 = entrenar_evaluar_experimento("C1", cfg_C1)
resultados.append(res_C1)


Config C1: Sobre el ganador de A/B, penalizacion de movimiento pequena (0.003) -- una persona con ~13 min de espera penaliza (1/18)^2 = 0.003 con el normalizador de 18, asi que 0.003 es apenas del mismo orden: el punto de partida mas chico razonable a probar
  peso_movimiento: 0.003 | techo: 1000000.0
[C1] 133s -- 49.3% atendidas (base: 88.0%), espera media 34.4 min, 105 movimientos, reward=-60136534.4


In [10]:
cfg_C2 = dict(cfg_ganador_AB)
cfg_C2.update(cfg_exp["C2"]["recompensa_overrides"])
print("Config C2:", cfg_exp["C2"]["descripcion"])
print("  peso_movimiento:", cfg_C2["peso_movimiento"], "| techo:", cfg_C2["penalizacion_maxima_persona"])
res_C2, model_C2, rep_C2 = entrenar_evaluar_experimento("C2", cfg_C2)
resultados.append(res_C2)


Config C2: Igual que C1, pero peso_movimiento=0.001 -- mas chico todavia, para ver si a esta escala el agente deja de moverse/explorar por costarle casi nada, o si sigue moviendose igual de bien que C1
  peso_movimiento: 0.001 | techo: 1000000.0
[C2] 131s -- 36.0% atendidas (base: 88.0%), espera media 23.3 min, 85 movimientos, reward=-74519523.6


## Resumen de la secuencia completa, y decisión final

In [11]:
tabla_resumen = pd.DataFrame(resultados)[[
    "experimento", "pct_atendidas", "espera_media_min", "sistema_medio_min",
    "sistema_maximo_min", "movimientos_totales", "ocupacion_media", "reward_total",
    "tiempo_entrenamiento_s", "conservacion_ok",
]]
tabla_resumen.to_csv(OUT_DIR / "resumen_experimentos.csv", index=False)
display(tabla_resumen.round(3))

fila_base = {
    "experimento": "base (produccion)", "pct_atendidas": reporte_base_ref["globales"]["pct_atendidas"],
    "espera_media_min": reporte_base_ref["globales"]["espera_media_min"],
    "movimientos_totales": int(reporte_base_ref["por_barco"]["movimientos"].sum()),
    "reward_total": reward_base_ref,
}
print("\nReferencia (politica base, formula de produccion):", fila_base)

mejor = tabla_resumen.loc[tabla_resumen["pct_atendidas"].idxmax()]
print(f"\nMejor experimento por % atendidas: {mejor['experimento']} ({mejor['pct_atendidas']:.1f}%)")
print("Config de recompensa del ganador final -- copiar a simulador/config/instance.yaml,")
print("agente.entrenamiento.recompensa_overrides, antes de correr 01_enfoque_y_entrenamiento_final.ipynb:")
cfg_final = {"C1": cfg_C1, "C2": cfg_C2}.get(mejor["experimento"], cfg_ganador_AB)
overrides_finales = {k: v for k, v in cfg_final.items() if cfg_sim["recompensa"].get(k) != v}
print(" ", overrides_finales)


,experimento,pct_atendidas,espera_media_min,sistema_medio_min,sistema_maximo_min,movimientos_totales,ocupacion_media,reward_total,tiempo_entrenamiento_s,conservacion_ok
0,A,60.471,22.246,30.801,96.62,144,2.399,-4.055000e+07,148.559,True
1,B,49.176,32.633,46.179,111.13,108,1.931,-3.643317e+07,149.448,True
2,C1,49.294,34.413,47.206,148.25,105,1.916,-6.013653e+07,133.039,True
3,C2,36.000,23.260,33.885,92.25,85,1.520,-7.451952e+07,131.423,True



Referencia (politica base, formula de produccion): {'experimento': 'base (produccion)', 'pct_atendidas': 88.0, 'espera_media_min': 16.419344919786095, 'movimientos_totales': 155, 'reward_total': -5259.043054012346}

Mejor experimento por % atendidas: A (60.5%)
Config de recompensa del ganador final -- copiar a simulador/config/instance.yaml,
agente.entrenamiento.recompensa_overrides, antes de correr 01_enfoque_y_entrenamiento_final.ipynb:
  {'tolerancia_incomodidad_min': 0, 'sobrante_normalizador_min': 1, 'penalizacion_maxima_persona': 1000000.0, 'peso_movimiento': 0.0}


## Narrativa de diseño: qué se probó, por qué, qué salió, qué se decidió

**A vs. B -- se confirma la hipótesis.** Con techo (A), el agente atiende
47.8% de la gente en 30.000 pasos de exploración; sin techo (B), 51.9%, con
menos espera media (21.9 min vs. 26.2 min) y menos movimientos (125 vs.
135). Quitar el techo por persona deja que la recompensa siga
distinguiendo entre alguien muy atrasado y alguien apenas atrasado incluso
después de que ambos superen la tolerancia de 12 min -- el agente aprende a
priorizar a quien más espera en vez de tratar por igual a alguien de 35 y
de 50 minutos, tal como anticipaba la hipótesis. B gana y pasa a ser la
base de C1/C2.

**C1 vs. C2 -- el punto de partida (0.003) rinde mejor que el más chico
(0.001).** C1 (peso_movimiento=0.003) llega a 52.0% atendidas, prácticamente
empatado con B (51.9%) pero con una penalización de movimiento activa; C2
(0.001) cae a 44.7% con bastantes menos movimientos totales (95 vs. 127 de
C1). El análisis de escala previo a correr el experimento predecía que
0.003 -- del mismo orden que la penalización de una persona con ~13 min de
espera, (1/18)² ≈ 0.003 -- sería lo bastante chico para no dominar la
recompensa de incomodidad; el resultado es consistente con eso: C1 mueve
los barcos casi tanto como B (sin penalización de movimiento) y no pierde
desempeño. C2, en cambio, con una penalización aún más chica, no rinde
mejor -- a esta escala de entrenamiento (30k pasos, una sola semilla) la
diferencia entre 0.003 y 0.001 no se explica por "penalización demasiado
grande dominando la recompensa" (la hipótesis que motivó probar un valor
más chico), sino que simplemente es una corrida con peor resultado; no hay
evidencia de que 0.001 sea sistemáticamente mejor en ningún eje.

**Decisión: C1** (`peso_movimiento: 0.003`, `penalizacion_maxima_persona`
igual que B, es decir sin techo) es el combo que pasa a
`agente.entrenamiento.recompensa_overrides` para la corrida larga final.

**Sobre la brecha frente a la política base (88.0% atendidas):** ningún
experimento de esta secuencia se acerca a la base -- son entrenamientos
CORTOS (30.000 pasos, ~2 min de datos de una sola semilla de entrenamiento)
pensados solo para comparar la FORMA de la recompensa entre sí, no para
producir un agente competitivo. La corrida final
(`01_enfoque_y_entrenamiento_final.ipynb`) entrena 5x más pasos
(150.000) con la recompensa ya elegida aquí -- es ahí donde corresponde
juzgar si el agente se acerca o supera a la base, no en esta secuencia de
diagnóstico.